**CellBender benchmarks**

Stephen Fleming

2026/08/11

# Data

In [ ]:
git_commit = "main"  # overridden by papermill
outputs_json = "{}"  # JSON string: {sample_name: gcs_output_dir}

Download files.

In [ ]:
import json as _json
import os
import subprocess

LOCAL_DATA_DIR = "/tmp/cellbender_benchmark_data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

outputs = _json.loads(outputs_json)

GCS_INPUT_BASE = "gs://broad-dsde-methods-sfleming/cellbender_test"
SAMPLE_INPUT_GCS = {
    "pbmc8k":  f"{GCS_INPUT_BASE}/pbmc8k_raw_gene_bc_matrices.h5",
    "rat6k":   f"{GCS_INPUT_BASE}/PCL_rat_A_LA6_raw_feature_bc_matrix.h5",
    "hgmm12k": f"{GCS_INPUT_BASE}/hgmm_12k_raw_gene_bc_matrices.h5",
    "pbmc5k":  f"{GCS_INPUT_BASE}/5k_pbmc_protein_v3_nextgem_raw_feature_bc_matrix.h5",
}

def _gcs_download(gcs_path, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    subprocess.run(["gsutil", "cp", gcs_path, local_dir + "/"], check=True)
    return os.path.join(local_dir, os.path.basename(gcs_path))

for sample, out_dir in outputs.items():
    if sample not in SAMPLE_INPUT_GCS:
        continue
    sample_local = os.path.join(LOCAL_DATA_DIR, sample)
    _gcs_download(SAMPLE_INPUT_GCS[sample], sample_local)
    _gcs_download(f"{out_dir.rstrip('/')}/{sample}_out.h5", sample_local)

pbmc_input = os.path.join(LOCAL_DATA_DIR, "pbmc8k", "pbmc8k_raw_gene_bc_matrices.h5")
pbmc_output = os.path.join(LOCAL_DATA_DIR, "pbmc8k", "pbmc8k_out.h5")

rat_heart_input = os.path.join(LOCAL_DATA_DIR, "rat6k", "PCL_rat_A_LA6_raw_feature_bc_matrix.h5")
rat_heart_output = os.path.join(LOCAL_DATA_DIR, "rat6k", "rat6k_out.h5")

hgmm_input = os.path.join(LOCAL_DATA_DIR, "hgmm12k", "hgmm_12k_raw_gene_bc_matrices.h5")
hgmm_output = os.path.join(LOCAL_DATA_DIR, "hgmm12k", "hgmm12k_out.h5")

citeseq_input = os.path.join(LOCAL_DATA_DIR, "pbmc5k", "5k_pbmc_protein_v3_nextgem_raw_feature_bc_matrix.h5")
citeseq_output = os.path.join(LOCAL_DATA_DIR, "pbmc5k", "pbmc5k_out.h5")

# Imports and helper functions

In [ ]:
import scanpy as sc
from scanpy.plotting._dotplot import DotPlot

make_scanpy_doplot = DotPlot._dotplot
import os
from typing import List, Optional

import anndata
import colorcet as cc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr as corr_pearson

from cellbender.remove_background.downstream import (
    load_anndata_from_input_and_output,
    load_anndata_from_input_and_outputs,
)

%matplotlib inline

In [ ]:
sc.set_figure_params(fontsize=14, vector_friendly=True)

In [ ]:
print(f'scanpy version is {sc.__version__}')
print(f'anndata version is {anndata.__version__}')

In [ ]:
def tick_label_str(val):
    if val >= 1e3:
        return f'{(val / 1000):.0f}k'
    else:
        return str(val)

In [ ]:
def qc_and_typical_workflow(adata: anndata.AnnData,
                            layer: str,
                            mito_layer: Optional[str] = 'cellranger',
                            umi_percentile: float = 95,
                            gene_percentile: float = 95,
                            n_gene_min: float = 0,
                            mito_frac_percentile: float = 90,
                            n_pcs: int = 25,
                            leiden_resolution: float = 0.5):
    """Do most basic cell QC and a typical scanpy workflow"""

    # calculate a few metrics and look at them
    if 'mito_frac' not in adata.obs.keys():
        print('Calculating mito_frac assuming mito genes start with "MT-"')
        print(f"{adata.var_names.str.startswith('MT-').sum()} mito genes")
        mat = adata.X if mito_layer is None else adata.layers[mito_layer]
        adata.obs['mito_frac'] = np.array(
            mat[:, adata.var_names.str.startswith('MT-')].sum(axis=1)
        ).squeeze() / np.array(mat.sum(axis=1) + 1e-10).squeeze()
        plt.hist(adata.obs['mito_frac'].values.flatten(), bins=100)
        plt.xlabel('mito_frac')
        plt.ylabel('n_cells')
        plt.show()
    adata.obs['n_umi'] = np.array(adata.X.sum(axis=1)).squeeze()
    adata.obs['n_gene'] = np.array((adata.X > 0).sum(axis=1)).squeeze()

    # get rid of outlier cells
    print(f'{adata.shape[0]} cells before cell QC')
    adata_qc = adata[(adata.obs['n_umi'] < np.percentile(adata.obs['n_umi'], q=umi_percentile))
                     & (adata.obs['n_gene'] < np.percentile(adata.obs['n_gene'], q=gene_percentile))
                     & (adata.obs['mito_frac'] <
                        np.percentile(adata.obs['mito_frac'], q=mito_frac_percentile))
                     & (adata.obs['n_gene'] >= n_gene_min)].copy()
    print(f'{adata_qc.shape[0]} cells after cell QC')

    # typical scanpy workflow
    adata_qc.X = adata_qc.layers[layer].copy()
    sc.pp.normalize_total(adata_qc)
    sc.pp.log1p(adata_qc)
    sc.pp.highly_variable_genes(adata_qc, layer=layer,
                                n_top_genes=2000, flavor='seurat_v3')
    adata_qc.obsm['hvgX'] = adata_qc.X[:, adata_qc.var['highly_variable']].copy()
    adata_qc.obsm['scaledX'] = sc.pp.scale(adata_qc.obsm['hvgX'],
                                           zero_center=True,
                                           max_value=10.,
                                           copy=True)
    adata_qc.obsm['X_pca'] = sc.tl.pca(adata_qc.obsm['scaledX'])
    sc.pp.neighbors(adata_qc,
                    use_rep='X_pca',
                    n_pcs=n_pcs,
                    method='umap',
                    metric='cosine',
                    n_neighbors=20)
    sc.tl.umap(adata_qc)
    sc.tl.leiden(adata_qc, resolution=leiden_resolution)

    return adata_qc

In [ ]:
def annotate_umap(adata, key='leiden', use_rep='X_umap', **kwargs):
    """Add annotations on top of a UMAP plot"""

    # annotate
    d = adata.obs[[key]].copy()
    d['x'] = adata.obsm[use_rep][:, 0]
    d['y'] = adata.obsm[use_rep][:, 1]
    for r in d.groupby(key).median().iterrows():
        plt.text(r[1][0], r[1][1], r[0], **kwargs)

# PBMC 8k

Data: "8k PBMCs from a Healthy Donor"

Provenance: 10x Genomics public data, v2 chemistry, CellRanger 2.1.0 analysis

Link: https://www.10xgenomics.com/resources/datasets/8-k-pbm-cs-from-a-healthy-donor-2-standard-2-1-0

In [ ]:
pbmc_leiden_resolution = 0.55

## c, d, e. CellBender data

In [ ]:
adata = load_anndata_from_input_and_output(
    input_file=pbmc_input,
    output_file=pbmc_output,
)
adata.var_names_make_unique()

# eliminate empty droplets as defined by cellbender
cell_prob_key = 'cell_probability'
if cell_prob_key not in adata.obs.keys():
    cell_prob_key = 'latent_cell_probability'
pbmc8k_cell_bcs = adata.obs_names[adata.obs[cell_prob_key] > 0.5]
adata = adata[pbmc8k_cell_bcs].copy()

adata.layers['cellbender'] = adata.X.copy()
adata

In [ ]:
adata_qc = qc_and_typical_workflow(adata,
                                   layer='cellbender',
                                   mito_layer='cellranger',
                                   leiden_resolution=pbmc_leiden_resolution,
                                   n_gene_min=100)

In [ ]:
sc.pl.embedding(adata_qc, basis='umap', color='leiden', show=False)
annotate_umap(adata_qc, fontsize=18)
plt.axis('off')
plt.title('')
plt.gca().get_legend().remove()

# savefig('pbmc8k_umap_cellbender.pdf')

plt.show()

In [ ]:
adata_qc

In [ ]:
adata_qc.var['n_cellranger_in_qced_cells'] = np.array(adata_qc.layers['cellranger'].sum(axis=0)).squeeze()
adata_qc.var['n_cellbender_in_qced_cells'] = np.array(adata_qc.layers['cellbender'].sum(axis=0)).squeeze()

In [ ]:
pbmc_markers = {
    'Monoctypes\n/ Neutrophils': ['S100A8', 'S100A9', 'S100A12'],
    'Monocytes\n/ pDCs': ['LYZ', 'CST3', 'FCN1'],
    'T': ['IL32', 'TRAC'],
    'T CD4+ Naive / TE': ['CCR7'],
    'Tregs': ['FOXP3'],
    'B': ['IGHD', 'CD79A'],
    'B naive': ['FCER2'],
    'B memory': ['TNFRSF13B'],
    'T CD8+': ['CD8A', 'CD8B'],
    'T Cytotoxic': ['NKG7', 'GNLY'],
    'T gamma delta': ['TRGC1'],
    'MAIT': ['SLC4A10'],
    'NK': ['KLRF1', 'SPON2'],
    'Monocytes NC / I': ['FCGR3A'],
    'Progenitor': ['PPBP'],
    'Baso. / Neutro.\n/ Progenitor': ['SDPR', 'CLU'],
    'pDCs': ['LILRA4'],
    '(broad expression)': ['PTPRC'],
}

In [ ]:
# overview of the comparison using the cellbender data for clustering

max_frac = 0.5

adata_qc.X = adata_qc.layers['cellranger'].copy()
sc.pp.normalize_total(adata_qc)

sc.pl.dotplot(
    adata_qc,
    var_names=pbmc_markers,
    groupby='leiden',
    log=True,
    vmin=-0.5,
    color_map='Blues',
    dot_max=max_frac,
)

adata_qc.X = adata_qc.layers['cellbender'].copy()
sc.pp.normalize_total(adata_qc)

sc.pl.dotplot(
    adata_qc,
    var_names=pbmc_markers,
    groupby='leiden',
    log=True,
    vmin=-0.5,
    color_map='Blues',
    dot_max=max_frac,
)

## Supplementary Table: differential expression

In [ ]:
cb_clusters_0_4 = adata_qc[adata_qc.obs['leiden'].isin(['0', '4'])].copy()

cb_clusters_0_4.X = cb_clusters_0_4.layers['cellbender'].copy()
sc.pp.normalize_total(cb_clusters_0_4)
sc.pp.log1p(cb_clusters_0_4)

In [ ]:
sc.tl.rank_genes_groups(cb_clusters_0_4, 'leiden', method='wilcoxon')

In [ ]:
df = sc.get.rank_genes_groups_df(cb_clusters_0_4, group='0')
df[[g in ['LYZ', 'CST3', 'S100A8', 'S100A9', 'PTPRC'] for g in df['names']]]

In [ ]:
cb_clusters_0_4.X = cb_clusters_0_4.layers['cellranger'].copy()
sc.pp.normalize_total(cb_clusters_0_4)
sc.pp.log1p(cb_clusters_0_4)

In [ ]:
sc.tl.rank_genes_groups(cb_clusters_0_4, 'leiden', method='wilcoxon')

In [ ]:
df_raw = sc.get.rank_genes_groups_df(cb_clusters_0_4, group='0')
df_raw[[g in ['LYZ', 'CST3', 'S100A8', 'S100A9', 'PTPRC'] for g in df_raw['names']]]

## Supplementary:  LYZ violin plots

Use the CellBender clustering

In [ ]:
# adata_raw_qc.obs['cellbender_leiden'] = adata_qc.obs['leiden'].copy()

In [ ]:
plt.figure(figsize=(5, 6))

adata_qc.X = adata_qc.layers['cellranger'].copy()
sc.pp.normalize_total(adata_qc)
sc.pp.log1p(adata_qc)

plt.subplot(2, 1, 1)
sc.pl.violin(adata_qc, keys='LYZ', groupby='leiden', ax=plt.gca(), show=False)
plt.ylabel('Log of normalized counts')
plt.xlabel('')#Leiden 0.5 cluster obtained from CellBender data')
plt.grid(False)
plt.gca().twinx()
plt.ylabel('LYZ raw')
plt.yticks([])
plt.grid(False)

adata_qc.X = adata_qc.layers['cellbender'].copy()
sc.pp.normalize_total(adata_qc)
sc.pp.log1p(adata_qc)

plt.subplot(2, 1, 2)
sc.pl.violin(adata_qc, keys='LYZ', groupby='leiden', ax=plt.gca(), show=False)
plt.ylabel('Log of normalized counts')
plt.xlabel('Leiden 0.5 cluster obtained from CellBender data')
plt.grid(False)
plt.gca().twinx()
plt.ylabel('LYZ after cellbender')
plt.yticks([])
plt.grid(False)

# savefig('supp_pbmc8k_LYZ_violins.pdf')

plt.show()

## Supplementary: UMAPs of genes

In [ ]:
# pick a few

top_removed_genes = ['S100A9',
 'S100A8',
 'FCER1G',
 'GNLY', 'NKG7', 'CST3',
 'IGKC',
 'LST1',
 'AIF1',
 'HLA-DRA',
 'FCN1',
 'LYZ']

In [ ]:
print('Colorbar axes are truncated at the 80th percentile')

for g in top_removed_genes:

    plt.figure(figsize=(6, 3))
    a = 1

    for layer in ['cellranger', 'cellbender']:
        adata_qc.X = adata_qc.layers[layer].copy()
        sc.pp.normalize_total(adata_qc)
        if 'log1p' in adata_qc.uns.keys():
            del adata_qc.uns['log1p']
        sc.pp.log1p(adata_qc)
        ax = plt.subplot(1, 2, a)
        if a == 1:
            gene_counts = np.array(adata_qc.X[:, adata_qc.var_names == g].todense()).squeeze()
            vmax = max(np.percentile(gene_counts, q=80), 0.5)
        sc.pl.embedding(adata_qc, basis='umap', color=g, color_map='Oranges',
                        show=False, vmin=0, vmax=vmax, ax=ax, edges_width=0)
        plt.title(f'{layer.replace("cellranger", "Raw data").replace("cellbender", "CellBender")}: {g}')
        a += 1

    plt.tight_layout()

#     savefig(f'pbmc8k_removal_umap_{g}.pdf')

    plt.show()

# Rat 6k: cell probabilities

Data: "10k left atrial cells from a healthy Wistar Rat"

Provenance: Precision Cardiology Lab data (Broad Institute, Bayer), v2 chemistry, CellRanger 3.1.0 analysis

Link: (single cell portal?)

Data download command (full CellRanger v3):
```bash
tbd {DATASET_DIR}/PCL_rat_A_LA6_raw_feature_bc_matrix.h5
```

Data download command (cells only, CellRanger v3):
```bash
tbd {DATASET_DIR}/PCL_rat_A_LA6_filtered_feature_bc_matrix_cellranger3.h5
```

Data download command (cells only, CellRanger v2):
```bash
tbd {DATASET_DIR}/PCL_rat_A_LA6_filtered_gene_bc_matrices_h5_cellranger2.h5
```

Cellbender data:

```bash
cellbender remove-background --cuda --input {DATASET_DIR}/PCL_rat_A_LA6_raw_feature_bc_matrix.h5 --output {CELLBENDER_DIR}/PCL_rat_A_LA6_20211021_out.h5 --expected-cells 8000 --total-droplets-included 25000
```

In [ ]:
# load dataset and annotate called cells

adata = load_anndata_from_input_and_output(
    input_file=rat_heart_input,
    output_file=rat_heart_output,
    input_layer_key='cellranger3',
)
adata.var_names_make_unique()

# cellbender cell calls
cell_prob_key = 'cell_probability'
if cell_prob_key not in adata.obs.keys():
    cell_prob_key = 'latent_cell_probability'
adata.obs['cell_cellbender'] = (adata.obs[cell_prob_key] > 0.5)

## Figure 1f. cell probability image

In [ ]:
# plot UMI curves with cell calls overlaid

plt.figure(figsize=(4, 4))

counts = adata.obs['n_cellranger3'].values
order = np.argsort(counts)[::-1]

# for each cell calling algorithm
for a, key in enumerate(['cellbender']):

    # UMI curve
    plt.semilogy(counts[order], color='black', lw=5, rasterized=True)
    plt.ylim([30, 25000])
    plt.ylabel('UMI count')
    plt.xlabel('Droplet ranked by count')
    plt.grid(False)
    ta = plt.gca().twinx()

    # cell probability
    cell_prob_key = 'cell_probability'
    if cell_prob_key not in adata.obs.keys():
        cell_prob_key = 'latent_cell_probability'
    plt.plot(adata.obs[cell_prob_key].values[order],
             '.', color='tab:red', ms=1, rasterized=True)
    plt.ylim([-0.02, 1.02])
    plt.yticks(color='tab:red')
    plt.ylabel('Inferred cell probability', color='tab:red')

    # labels
    ta.grid(False)
    ta.set_xlim([-1000, 25000])
    xticks = [0, 10000, 20000]
    ta.set_xticks(xticks)
    ta.set_xticklabels([tick_label_str(v) for v in xticks])

plt.tight_layout()

# savefig('PCL_rat_A_LA6_cell_probability_thumbnail.pdf')

plt.show()

## a. UMI curve

In [ ]:
# plot UMI curves with cell calls overlaid, in a different format

fig = plt.figure(figsize=(5, 4))

counts = adata.obs['n_cellranger3'].values
order = np.argsort(counts)[::-1]
bins = 50
key = 'cellbender'

# UMI curve
plt.semilogy(counts[order], color='black')
plt.grid(False)
twinned_ax = plt.gca().twinx()

# binned ratio of cells and empties
xbinedges = np.linspace(start=0, stop=adata.shape[0], num=bins, dtype=int)
xbinwidth = xbinedges[1] - xbinedges[0]
n_cells_bin = []
n_empties_bin = []
fraction_cells_bin = []
bin_centers = []
for start, stop in zip(xbinedges[:-1], xbinedges[1:]):
    n_cells = adata.obs[f'cell_{key}'][order][start:stop].sum()
    n_empties = (~adata.obs[f'cell_{key}'][order][start:stop]).sum()
    n_cells_bin.append(n_cells)
    n_empties_bin.append(n_empties)
    fraction_cells_bin.append(n_cells / (n_cells + n_empties))
    bin_centers.append(np.mean([start, stop]))

twinned_ax.bar(bin_centers, fraction_cells_bin,
                xbinwidth, label='cell', alpha=0.4, color='lightgreen')
twinned_ax.bar(bin_centers, [1. - f for f in fraction_cells_bin],
                xbinwidth, bottom=fraction_cells_bin,
                label='not cell', alpha=0.1, color='lightgray')
twinned_ax.set_ylim([0, 1])
twinned_ax.set_ylabel('Cell fraction', color='green')
twinned_ax.set_yticklabels(labels=[f'{v:.1f}' for v in twinned_ax.get_yticks()],
                            color='green')
twinned_ax.grid(False)

# labels
plt.text(x=13000, y=10000, s=key)
plt.xlim([-200, 25000])
xticks = [0, 5000, 10000, 15000, 20000, 25000]
plt.xticks(xticks, [tick_label_str(v) for v in xticks])
plt.ylim([50, 25000])
plt.xlabel('Droplet ranked by counts')
plt.ylabel('UMI count')

plt.tight_layout()

# savefig('PCL_rat_A_LA6_cell_calling_comparison.pdf')

plt.show()

# HGMM 12k

Data: "12k 1:1 Mixture of Fresh Frozen Human (HEK293T) and Mouse (NIH3T3) Cells"

Provenance: 10x Genomics public data, v2 chemistry, CellRanger 2.1.0 analysis

Link: https://www.10xgenomics.com/resources/datasets/12-k-1-1-mixture-of-fresh-frozen-human-hek-293-t-and-mouse-nih-3-t-3-cells-2-standard-2-1-0

In [ ]:
# load input file and all the relevant outputs as layers

adata_hgmm = load_anndata_from_input_and_outputs(
    input_file=hgmm_input,
    output_files={
        'cellbender_0.01': hgmm_output,
    },
)

# assign species labels to genes

adata_hgmm.var['species'] = ['human' if g.startswith('hg19_') else 'mouse'
                             for g in adata_hgmm.var['gene_id']]

adata_hgmm

In [ ]:
mouse_genes = (adata_hgmm.var['species'] == 'mouse')
human_genes = (adata_hgmm.var['species'] == 'human')

mouse_counts = np.array(adata_hgmm.layers['cellranger'][:, mouse_genes].sum(axis=1)).squeeze()
human_counts = np.array(adata_hgmm.layers['cellranger'][:, human_genes].sum(axis=1)).squeeze()

cutoff = 750

non_doublet_logic = ~((mouse_counts > cutoff)
                      & (human_counts > cutoff))

In [ ]:
# calculate stats on cross-species counts in each output

def get_cross_species_stats(adata, layer, contam_gene_logic):

    contam_counts_per_cell = np.array(adata.layers[layer][:, contam_gene_logic].sum(axis=1)).squeeze()
    q1 = np.percentile(contam_counts_per_cell, q=25)
    q2 = np.percentile(contam_counts_per_cell, q=50)
    q3 = np.percentile(contam_counts_per_cell, q=75)

    total_counts_per_cell = np.array(adata.layers[layer].sum(axis=1)).squeeze()
    num_zeros = np.sum(total_counts_per_cell == 0)
    if num_zeros > 0:
        print(f'Warning: {layer} has {num_zeros} zero-count cells (out of {len(total_counts_per_cell)}) that had {cutoff} counts in cellranger')
    nonzeros = (total_counts_per_cell > 0)
    q1_frac = np.percentile(contam_counts_per_cell[nonzeros] / total_counts_per_cell[nonzeros], q=25)
    q2_frac = np.percentile(contam_counts_per_cell[nonzeros] / total_counts_per_cell[nonzeros], q=50)
    q3_frac = np.percentile(contam_counts_per_cell[nonzeros] / total_counts_per_cell[nonzeros], q=75)

    return {'short': {'count_q1': [q1], 'count_q2': [q2], 'count_q3': [q3],
                      'frac_q1': [q1_frac], 'frac_q2': [q2_frac], 'frac_q3': [q3_frac]},
            'long': {'count': contam_counts_per_cell[nonzeros],
                     'frac': contam_counts_per_cell[nonzeros] / total_counts_per_cell[nonzeros]}}

df_long = pd.DataFrame(columns=['layer', 'contaminant_species', 'count', 'frac'])
df = pd.DataFrame(columns=['layer', 'contaminant_species',
                           'count_q1', 'count_q2', 'count_q3',
                           'frac_q1', 'frac_q2', 'frac_q3'])

# layers = ['cellranger', 'cellbender_0.01', 'cellbender_0.05', 'cellbender_0.1', 'decontx']
layers = ['cellranger', 'cellbender_0.01']

print('mouse cells')
for layer in layers:
    stats = get_cross_species_stats(
        adata_hgmm[non_doublet_logic & (mouse_counts > cutoff)],  # mouse cells
        layer=layer,
        contam_gene_logic=human_genes,
    )
    stats['short'].update({'layer': layer, 'contaminant_species': 'human'})
    stats['long'].update({'layer': np.array([layer] * len(stats['long']['count'])),
                          'contaminant_species': np.array(['human'] * len(stats['long']['count']))})
    df = pd.concat([df, pd.DataFrame(data=stats['short'])], axis=0)
    df_long = pd.concat([df_long, pd.DataFrame(data=stats['long'])], axis=0)

print('human cells')
for layer in layers:
    stats = get_cross_species_stats(
        adata_hgmm[non_doublet_logic & (human_counts > cutoff)],  # human cells
        layer=layer,
        contam_gene_logic=mouse_genes,
    )
    stats['short'].update({'layer': layer, 'contaminant_species': 'mouse'})
    stats['long'].update({'layer': [layer] * len(stats['long']['count']),
                          'contaminant_species': ['mouse'] * len(stats['long']['count'])})
    df = pd.concat([df, pd.DataFrame(data=stats['short'])], axis=0)
    df_long = pd.concat([df_long, pd.DataFrame(data=stats['long'])], axis=0)

df

In [ ]:
import seaborn as sns

plt.figure(figsize=(1.75, 3))
sns.boxplot(data=df_long,
            x='layer',
            y='frac',
            hue='contaminant_species',
            fliersize=0,
            ax=plt.gca(),
            width=0.5)
plt.yscale('log')
plt.xticks(range(2),
           labels=['Raw', 'FPR 0.01'],
           rotation=80)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title='Contaminant')
plt.ylabel('Fraction of cross-species\ncounts per cell')
plt.xlabel('')
plt.ylim([1e-5, 1e-1])
plt.yticks([1e-5, 1e-4, 1e-3, 1e-2, 1e-1])

# savefig('hgmm12k_removal_quantification_comparison_frac.pdf')

plt.show()

In [ ]:
adata_hgmm.var['n_cellbender'] = np.array(adata_hgmm.layers['cellbender_0.01'].sum(axis=0)).squeeze()
adata_hgmm.var['n_cellranger'] = np.array(adata_hgmm.layers['cellranger'].sum(axis=0)).squeeze()
adata_hgmm.var['frac_remaining'] = adata_hgmm.var['n_cellbender'] / adata_hgmm.var['n_cellranger']

plt.figure(figsize=(4, 2))
plt.semilogx(adata_hgmm.var['n_cellranger'].values, adata_hgmm.var['frac_remaining'].values, 'k.', ms=2)
val = np.mean(adata_hgmm.var['frac_remaining'].values[adata_hgmm.var['n_cellranger'].values > 1e3])
plt.plot([1e0, 1e7], [val, val], color='red', lw=1)
plt.grid(False)
plt.xlabel('Total raw gene counts')
plt.ylabel('Fraction remaining')
plt.title('Per-gene removal')
plt.show()

In [ ]:
# plot the cross-species counts, before and after

def linear_mixed_species(adata,
                         gene_logic1: np.ndarray,
                         gene_logic2: np.ndarray,
                         layer_keys: List[str],
                         colors: Optional[List] = None,
                         raw_key: str = 'cellranger',
                         raw_color: str = 'k',
                         figsize=(5, 5),
                         add_one: bool = False,
                         show_empties: bool = False):
    """Utility for a mixed-species plot"""

    cell_prob_key = 'cell_probability'
    if cell_prob_key not in adata.obs.keys():
        cell_prob_key = 'latent_cell_probability'
    cell = (adata.obs[cell_prob_key] > 0.5)

    val = 1 if add_one else 0

    plt.figure(figsize=figsize)

    if show_empties:
        plt.plot(np.array(adata.layers[raw_key]
                          [~cell][:, gene_logic1].sum(axis=1)).squeeze() + val,
                 np.array(adata.layers[raw_key]
                          [~cell][:, gene_logic2].sum(axis=1)).squeeze() + val,
                 '.', color='lightgray', ms=1, label='Raw (empty)', rasterized=True)

    plt.plot(np.array(adata.layers[raw_key]
                     [cell][:, gene_logic1].sum(axis=1)).squeeze() + val,
             np.array(adata.layers[raw_key]
                     [cell][:, gene_logic2].sum(axis=1)).squeeze() + val,
             '.', color=raw_color, ms=1, label='Raw (non-empty)', rasterized=True)

    if colors is not None:
        assert len(colors) >= len(layer_keys), 'Need at least as many colors as layers'
        for c, key in zip(colors,
                          layer_keys):
            plt.plot(np.array(adata.layers[key]
                              [cell][:, gene_logic1].sum(axis=1)).squeeze() + val,
                     np.array(adata.layers[key]
                              [cell][:, gene_logic2].sum(axis=1)).squeeze() + val,
                     '.', color=c, ms=1,
                     label=f'CB: FPR {key.split("_")[1]}' if '_' in key else f'{key}',
                     rasterized=True)
    else:
        for key in layer_keys:
            plt.plot(np.array(adata.layers[key]
                              [cell][:, gene_logic1].sum(axis=1)).squeeze() + val,
                     np.array(adata.layers[key]
                              [cell][:, gene_logic2].sum(axis=1)).squeeze() + val,
                     '.', ms=1,
                     label=f'CB: FPR {key.split("_")[1]}' if '_' in key else f'{key}',
                     rasterized=True)

    # make a legend
    lgnd = plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    for h in lgnd.legend_handles:
        h.set_markersize(10)
    plt.grid(False)


def marginal_histogram(
    adata,
    gene_logic1: np.ndarray,
    gene_logic2: np.ndarray,
    layer_keys: List[str],
    which_histogram: str,  # 'x' or 'y'
    log_bins: bool = True,
    n_bins: int = 50,
    xxmax: float = 1e5,
    colors: Optional[List] = None,
    raw_key: str = 'cellranger',
    raw_color: str = 'k',
    figsize=(5, 2),
    add_one: bool = True,
    show_empties: bool = False,
):
    """Utility for a mixed-species plot's marginal histogram"""

    if log_bins:
        assert add_one, 'if log_bins is True, add_one should be also'

    cell_prob_key = 'cell_probability'
    if cell_prob_key not in adata.obs.keys():
        cell_prob_key = 'latent_cell_probability'
    cell = (adata.obs[cell_prob_key] > 0.5)

    val = 1 if add_one else 0

    if which_histogram == 'x':
        gene_logic = gene_logic1
    else:
        assert which_histogram == 'y', 'which_histogram must be in ["x", "y"]'
        gene_logic = gene_logic2

    xx = (([0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
           + [a for a in np.logspace(0, np.log10(xxmax), n_bins, base=10)
              if a > 6.5]) if log_bins
          else np.linspace(0, xxmax, n_bins))

    def _hist(data, color, label):
        plt.hist(data, bins=xx, histtype='step',
                 color=color, label=label, rasterized=True,
                 density=True,
                 lw=2 if ('0.1' in label) else 1,
                 orientation='vertical' if (which_histogram == 'x') else 'horizontal')

    plt.figure(figsize=figsize)

    if show_empties:
        _hist(np.array(adata.layers[raw_key]
                       [~cell][:, gene_logic].sum(axis=1)).squeeze() + val,
              color='lightgray', label='Raw (empty)')

    _hist(np.array(adata.layers[raw_key]
                   [cell][:, gene_logic].sum(axis=1)).squeeze() + val,
          color=raw_color, label='Raw')

    if colors is not None:
        assert len(colors) >= len(layer_keys), 'Need at least as many colors as layers'
        for c, key in zip(colors,
                          layer_keys):
            _hist(np.array(adata.layers[key]
                           [cell][:, gene_logic].sum(axis=1)).squeeze() + val,
                  color=c,
                  label=f'CB: FPR {key.split("_")[1]}' if '_' in key else f'{key}')
    else:
        for key in layer_keys:
            _hist(np.array(adata.layers[key]
                           [cell][:, gene_logic].sum(axis=1)).squeeze() + val,
                  label=f'CB: FPR {key.split("_")[1]}' if '_' in key else f'{key}')

    if log_bins:
        if which_histogram == 'x':
            plt.xscale('log')
            plt.xticks(ticks=[1, 2, 10, 1e2, 1e3, 1e4, 1e5],
                       labels=[1, 2, '$10^1$', '$10^2$', '$10^3$', '$10^4$', '$10^5$'])
        else:
            plt.yscale('log')
            plt.yticks(ticks=[1, 2, 10, 1e2, 1e3, 1e4, 1e5],
                       labels=[1, 2, '$10^1$', '$10^2$', '$10^3$', '$10^4$', '$10^5$'])

    # make a legend
    lgnd = plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.grid(False)

In [ ]:
# log-log plot for cellbender

linear_mixed_species(adata_hgmm[non_doublet_logic],
                     human_genes,
                     mouse_genes,
                     layer_keys=['cellbender_0.01'],  # , 'cellbender_0.05', 'cellbender_0.1'],
                     colors=['tab:green', 'tab:orange', 'tab:red'],
                     raw_key='cellranger',
                     raw_color='lightgray',
                     add_one=True,
                     figsize=(4, 4),
                     show_empties=False)
plt.xscale('log')
plt.yscale('log')
plt.xticks([1e0, 1e1, 1e2, 1e3, 1e4, 1e5])
plt.yticks([1e0, 1e1, 1e2, 1e3, 1e4, 1e5])
plt.xlim([0.6, 1e5])
plt.ylim([0.6, 1e5])
plt.xlabel('Human counts + 1')
plt.ylabel('Mouse counts + 1')
lgnd = plt.legend(loc='lower left', fontsize=12, frameon=False)
for h in lgnd.legend_handles:
    h.set_markersize(10)

# savefig('hgmm12k_species_removal_log.pdf')

plt.show()

In [ ]:
marginal_histogram(
    adata_hgmm[non_doublet_logic & (mouse_counts > cutoff)],
    human_genes,
    mouse_genes,
    layer_keys=['cellbender_0.01'],  # , 'cellbender_0.05', 'cellbender_0.1'],
    which_histogram='x',
    colors=['tab:green', 'tab:orange', 'tab:red'],
    raw_key='cellranger',
    raw_color='gray',
    add_one=True,
    figsize=(4, 2),
    show_empties=False,
)
plt.xlim([0.6, 1e5])
plt.gca().xaxis.tick_top()
plt.gca().xaxis.set_label_position('top')
plt.ylabel('Fraction mouse cells')
plt.xlabel('Human counts + 1')
plt.yscale('log')
plt.ylim(bottom=1e-4)

# savefig('hgmm12k_species_removal_marginal_hist_mousecells.pdf')

plt.show()

marginal_histogram(
    adata_hgmm[non_doublet_logic & (human_counts > cutoff)],
    human_genes,
    mouse_genes,
    layer_keys=['cellbender_0.01'],  # , 'cellbender_0.05', 'cellbender_0.1'],
    which_histogram='y',
    colors=['tab:green', 'tab:orange', 'tab:red'],
    raw_key='cellranger',
    raw_color='gray',
    add_one=True,
    figsize=(2, 4),
    show_empties=False,
)
plt.minorticks_off()
plt.ylim([0.6, 1e5])
plt.gca().yaxis.tick_right()
plt.gca().yaxis.set_label_position('right')
plt.gca().get_legend().remove()
plt.xlabel('Fraction human cells')
plt.ylabel('Mouse counts + 1')
plt.xscale('log')
plt.xlim(left=1e-4)

# savefig('hgmm12k_species_removal_marginal_hist_humancells.pdf')

plt.show()

In [ ]:
# linear plots for cellbender

# zoom in on mouse cells

linear_mixed_species(adata_hgmm[non_doublet_logic & (mouse_counts > cutoff)],
                     human_genes,
                     mouse_genes,
                     layer_keys=['cellbender_0.01'],  # , 'cellbender_0.05', 'cellbender_0.1'],
                     colors=['tab:green', 'tab:orange', 'tab:red'],
                     raw_key='cellranger',
                     raw_color='lightgray',
                     figsize=(3, 4),
                     show_empties=False)

yticks = [0, 1e4, 2e4, 3e4, 4e4]
plt.yticks(ticks=yticks, labels=[tick_label_str(v) for v in yticks])
plt.xlim([-20, 700])
plt.ylim([-1000, 30000])
plt.xlabel('Human counts')
plt.ylabel('Mouse counts')
plt.title('Mouse cells', color='darkcyan')
for axis_part in ['bottom', 'top', 'left', 'right']:
    plt.gca().spines[axis_part].set_color('darkcyan')

# savefig('hgmm12k_species_removal_linear_zoom_mouse.pdf')

plt.show()

# zoom in on human cells

linear_mixed_species(adata_hgmm[non_doublet_logic & (human_counts > cutoff)],
                     human_genes,
                     mouse_genes,
                     layer_keys=['cellbender_0.01'],  # , 'cellbender_0.05', 'cellbender_0.1'],
                     colors=['tab:green', 'tab:orange', 'tab:red'],
                     raw_key='cellranger',
                     raw_color='lightgray',
                     figsize=(4, 2.5),
                     show_empties=False)

xticks = [0, 2e4, 4e4, 6e4]
plt.xticks(ticks=xticks, labels=[tick_label_str(v) for v in xticks])
plt.xlim([-1000, 50000])
plt.ylim([-20, 500])
plt.xlabel('Human counts')
plt.ylabel('Mouse counts')
plt.title('Human cells', color='darkmagenta')
for axis_part in ['bottom', 'top', 'left', 'right']:
    plt.gca().spines[axis_part].set_color('darkmagenta')

# savefig('hgmm12k_species_removal_linear_zoom_human.pdf')

plt.show()

# PBMC 5k: CITE-seq as benchmark

## PBMC 5k with surface proteins

Data: "5k Peripheral blood mononuclear cells (PBMCs) from a healthy donor with cell surface proteins (Next GEM)"

Provenance: 10x Genomics public data, v3 Next GEM chemistry, CellRanger 3.1.0 analysis

Link: https://www.10xgenomics.com/resources/datasets/5-k-peripheral-blood-mononuclear-cells-pbm-cs-from-a-healthy-donor-with-cell-surface-proteins-next-gem-3-1-standard-3-1-0

In [ ]:
leiden_res_pbmc_5k = 0.6
min_n_gene = 300

In [ ]:
adata = load_anndata_from_input_and_output(
    input_file=citeseq_input,
    output_file=citeseq_output,
    input_layer_key='cellranger',
)
adata.var_names_make_unique()

# eliminate empty droplets as defined by cellbender
cell_prob_key = 'cell_probability'
if cell_prob_key not in adata.obs.keys():
    cell_prob_key = 'latent_cell_probability'
pbmc5k_cell_bcs = adata.obs_names[adata.obs[cell_prob_key] > 0.5]
adata = adata[pbmc5k_cell_bcs].copy()

adata.layers['cellbender'] = adata.X.copy()
adata

In [ ]:
len(adata.var[adata.var['feature_type'] == 'Antibody Capture'])

In [ ]:
def gene_from_antibody(antibody):
    """How to translate the antibody names into the 
    corresponding gene expression features.
    """

    # looked up manually, mostly on genecards.org
    lookups = {'HLA-DR': 'HLA-DRA',
               'PD-1': 'PDCD1',
               'CD3': 'CD3G',
               'CD11B': 'ITGAM',
               'CD15': 'FUT4',
               'CD16': 'FCGR3A',
               'CD20': 'MS4A1',
               'CD25': 'IL2RA',
               'CD56': 'NCAM1',
               'CD62L': 'SELL',
               'CD127': 'IL7R',
               'CD137': 'TNFRSF9',
               'CD197': 'CCR7',
               'CD278': 'ICOS',
               'CD335': 'NCR1',
               'CD45RA': 'PTPRC',
               'CD45RO': 'PTPRC'}

    antibody = antibody.upper()
    if antibody in lookups.keys():
        return lookups[antibody]
    elif antibody in adata.var_names:
        return antibody
    elif ((antibody[-1] not in [str(i) for i in range(10)])
        and (antibody[:-1] in adata.var_names)):
        return antibody[:-1]
    elif antibody.startswith('IGG'):
        ab = antibody.replace('IGG', 'IGHG')
        if ab in adata.var_names:
            return ab
        elif ab[:-1] in adata.var_names:
            return ab[:-1]
    raise ValueError(f'{antibody} not present...')


for anti in adata.var[adata.var['feature_type'] == 'Antibody Capture']['gene_id']:
    try:
        gene_from_antibody(anti)
    except ValueError:
        print(anti + ' is missing')

In [ ]:
matched_antibody_features = []

for ab in adata.var[adata.var['feature_type'] == 'Antibody Capture'].iterrows():
    matched_antibody_features.append(gene_from_antibody(ab[-1]['gene_id']))
    matched_antibody_features.append(ab[0])
# matched_antibody_features

In [ ]:
adata_qc = qc_and_typical_workflow(adata,
                                   layer='cellbender',
                                   leiden_resolution=leiden_res_pbmc_5k,
                                   n_gene_min=min_n_gene)

In [ ]:
# now with surface proteins

pbmc_markers = {
    'Monoctypes\n/ Neutrophils': ['S100A8', 'S100A9', 'S100A12', 'CD14', 'CD14_TotalSeqB'],
    'Monocytes\n/ pDCs': ['LYZ', 'CST3', 'FCN1'],
    'T': ['IL32', 'TRAC'],
    'T CD4+ Naive / TE': ['CCR7'],
    'Tregs': ['FOXP3'],
    'B': ['IGHD', 'CD79A'],
    'B naive': ['FCER2'],
    'B memory': ['TNFRSF13B'],
    'T CD8+': ['CD8A', 'CD8a_TotalSeqB', 'CD8B'],
    'T Cytotoxic': ['NKG7', 'GNLY'],
    'T gamma delta': ['TRGC1'],
    'MAIT': ['SLC4A10'],
    'NK': ['KLRF1', 'SPON2'],
    'Monocytes NC / I': ['FCGR3A', 'CD16_TotalSeqB'],
    'Progenitor': ['PPBP'],
    'Baso. / Neutro.\n/ Progenitor': ['CAVIN2', 'CLU'],
    'pDCs': ['LILRA4'],
    '(broad expression)': ['PTPRC', 'CD45RA_TotalSeqB', 'CD45RO_TotalSeqB'],
}

In [ ]:
f = sc.pl.dotplot(
    adata_qc,
    var_names=pbmc_markers,
    layer='cellbender',
    groupby='leiden',
    log=True,
    vmin=-0.5,
    color_map='Blues',
    show=False,
)
# for tick in f['mainplot_ax'].get_xticklabels():
#     tick.set_rotation(70)
plt.show()

In [ ]:
f = sc.pl.dotplot(
    adata_qc,
    var_names=pbmc_markers,
    layer='cellranger',
    groupby='leiden',
    log=True,
    vmin=-0.5,
    color_map='Blues',
    show=False,
)
# for tick in f['mainplot_ax'].get_xticklabels():
#     tick.set_rotation(70)
plt.show()

In [ ]:
adata_raw = sc.read_10x_h5(os.path.join(DATASET_DIR, '5k_pbmc_protein_v3_nextgem_raw_feature_bc_matrix.h5'),
                           gex_only=False)
adata_raw.var_names_make_unique()

# eliminate empty droplets as defined by cellbender
adata_raw = adata_raw[pbmc5k_cell_bcs].copy()

adata_raw.layers['cellranger'] = adata_raw.X.copy()
adata_raw

In [ ]:
adata_raw.X

In [ ]:
adata_raw_qc = qc_and_typical_workflow(adata_raw,
                                       layer='cellranger',
                                       leiden_resolution=leiden_res_pbmc_5k,
                                       n_gene_min=min_n_gene)

In [ ]:
f = sc.pl.dotplot(
    adata_raw_qc,
    var_names=pbmc_markers,
    groupby='leiden',
    log=True,
    vmin=-0.5,
    color_map='Blues',
    show=False,
)
# for tick in f['mainplot_ax'].get_xticklabels():
#     tick.set_rotation(70)
plt.show()

In [ ]:
# just plot the log2 of the ratio of one isoform to another, with a divergent colormap

adata_qc.X = adata_qc.layers['cellbender'].copy()
sc.pp.normalize_total(adata_qc)
sc.pp.log1p(adata_qc)
adata_qc.layers['cellbender_norm'] = adata_qc.X.copy()

adata_qc.X = adata_qc.layers['cellranger'].copy()
sc.pp.normalize_total(adata_qc)
sc.pp.log1p(adata_qc)
adata_qc.layers['cellranger_norm'] = adata_qc.X.copy()

cr_ro_vals = np.array(adata_qc.layers['cellranger']
                   [:, adata_qc.var_names == 'CD45RO_TotalSeqB'].todense()).squeeze()
cr_ra_vals = np.array(adata_qc.layers['cellranger']
                   [:, adata_qc.var_names == 'CD45RA_TotalSeqB'].todense()).squeeze()

cb_ro_vals = np.array(adata_qc.layers['cellbender']
                   [:, adata_qc.var_names == 'CD45RO_TotalSeqB'].todense()).squeeze()
cb_ra_vals = np.array(adata_qc.layers['cellbender']
                   [:, adata_qc.var_names == 'CD45RA_TotalSeqB'].todense()).squeeze()

adata_qc.obs['log_ratio_CD45RA_CD45RO_cellranger'] = np.log2(cr_ra_vals / (cr_ro_vals + 1e-4) + 1e-4)
adata_qc.obs['log_ratio_CD45RA_CD45RO_cellbender'] = np.log2(cb_ra_vals / (cb_ro_vals + 1e-4) + 1e-4)

for g in ['log_ratio_CD45RA_CD45RO_cellranger', 'log_ratio_CD45RA_CD45RO_cellbender']:

    print('cellranger, normalized and log scaled')
    plt.figure(figsize=(3, 3))
    sc.pl.embedding(adata_qc, basis='umap', vmin=-15, vmax=15, ax=plt.gca(),
                    color=g, color_map='RdYlGn', s=10, alpha=0.75, show=False)
    plt.title(('CellBender' if ('cellbender' in g) else 'Raw') + ':\nlog2 ( CD45RA / CD45RO )')
#     savefig(f'pbmc5k_{g}.pdf')
    plt.show()

### CD20 UMAP

In [ ]:
cd20_features = ['MS4A1', 'CD20_TotalSeqB']

print('cellranger, normalized and log scaled')
sc.pl.embedding(adata_qc, basis='umap', layer='cellranger_norm',
                color=cd20_features, color_map='Oranges', vmin=0)#, vmax=20)

print('cellbender, normalized and log scaled')
sc.pl.embedding(adata_qc, basis='umap', layer='cellbender_norm',
                color=cd20_features, color_map='Oranges', vmin=0)#, vmax=20)

### b. CITEseq after CellBender

In [ ]:
# this is FPR 0.1

f = sc.pl.dotplot(
    adata_qc,
    var_names=matched_antibody_features,
    groupby='leiden',
    log=True,
    vmin=-0.5,
    color_map='Blues',
    colorbar_title='Log expression\nin group',
    var_group_rotation=0,
    show=False,
)
for tick in f['mainplot_ax'].get_xticklabels():
    if tick.get_text().endswith('_TotalSeqB'):
        tick.set_color('red')

# savefig('pbmc5k_citeseq_dotplot_cellbender_0.1.pdf')

plt.show()

Will skip genes with very low expression, since there is not much to show or benchmark in those cases.

- CD34 with max expression 0.0016538869822397828

- CD80 with max expression 0.008617527782917023

- TNFRSF9 with max expression 0.023860281333327293

- CD274 with max expression 0.02070119045674801

- PDCD1 with max expression 0.04665709286928177

In [ ]:
# also re-order them so that genes with similar expression patterns are close

matched_antibody_features_expressed = [
    'ITGAM', 'CD11b_TotalSeqB',
    'CD14', 'CD14_TotalSeqB',
    'CD86', 'CD86_TotalSeqB',

    'CD4', 'CD4_TotalSeqB',
    'PTPRC', 'CD45RA_TotalSeqB', 'CD45RO_TotalSeqB',
    'SELL', 'CD62L_TotalSeqB',

    'CD3G','CD3_TotalSeqB',
    'CD27', 'CD27_TotalSeqB',
    'CD28', 'CD28_TotalSeqB',
    'IL7R', 'CD127_TotalSeqB',
    'ICOS', 'CD278_TotalSeqB',

    'TIGIT', 'TIGIT_TotalSeqB',
    'NCR1', 'CD335_TotalSeqB',
    'NCAM1', 'CD56_TotalSeqB',
    'FCGR3A', 'CD16_TotalSeqB',

    'CD19', 'CD19_TotalSeqB',
    'MS4A1', 'CD20_TotalSeqB',
    'HLA-DRA', 'HLA-DR_TotalSeqB',
    'IL2RA', 'CD25_TotalSeqB',

    'FUT4', 'CD15_TotalSeqB',

    'CD8A', 'CD8a_TotalSeqB',
    'CCR7', 'CD197_TotalSeqB',
    'SELL', 'CD62L_TotalSeqB',

    'CD69', 'CD69_TotalSeqB',
]

In [ ]:
# do the dotplot computation using scanpy

sc.set_figure_params(fontsize=18, vector_friendly=True)

dp = sc.pl.dotplot(
    adata_qc,
    var_names=matched_antibody_features_expressed,
    groupby='leiden',
    layer='cellbender',
    log=True,
    vmin=-0.5,
    color_map='Blues',
    colorbar_title='Log expression\nin group',
    var_group_rotation=0,
    dot_min=0,
    dot_max=1,
    show=False,
    return_fig=True,
)

# dataframes of dot size and dot color
dc = dp.dot_color_df.copy()
ds = dp.dot_size_df.copy()

# add in empty columns at the appropriate places to split things up
for i in reversed([2, 4, 6, 8, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47]):
    ds.insert(loc=i, column='', value=np.nan, allow_duplicates=True)
    dc.insert(loc=i, column='', value=np.nan, allow_duplicates=True)

col_ab = [c for c in ds.columns.tolist() if c.endswith('_TotalSeqB')]
col_not_ab = [c for c in ds.columns.tolist() if not c.endswith('_TotalSeqB')]
col_gene = [c for c in ds.columns.tolist() if ((not c.endswith('_TotalSeqB')) and (c != ''))]
col_not_gene = [c for c in ds.columns.tolist() if not ((not c.endswith('_TotalSeqB')) and (c != ''))]

# to compute "normalize"
plt.figure(figsize=(7, 4))
normalize_ab, _, _ = make_scanpy_doplot(dot_size=ds[col_ab],
                                        dot_color=dc[col_ab],
                                        dot_ax=plt.gca(),
                                        vmin=0,
                                        vmax=2.21,  # max val
                                        cmap='Reds')
plt.show()

plt.figure(figsize=(7, 4))
normalize_gene, _, _ = make_scanpy_doplot(dot_size=ds[col_gene],
                                          dot_color=dc[col_gene],
                                          dot_ax=plt.gca(),
                                          vmin=0,
                                          vmax=1.44,  # max val
                                          cmap='Blues')
plt.show()

# the combined plot
plt.figure(figsize=(20, 4))
ds_only_ab = ds.copy()
ds_only_ab[col_not_ab] = 0
dc_only_ab = dc.copy()
dc_only_ab[col_not_ab] = 0
make_scanpy_doplot(dot_size=ds_only_ab,
                   dot_color=dc_only_ab,
                   dot_ax=plt.gca(),
                   cmap='Reds')

ds_only_gene = ds.copy()
ds_only_gene[col_not_gene] = 0
dc_only_gene = dc.copy()
dc_only_gene[col_not_gene] = 0
make_scanpy_doplot(dot_size=ds_only_gene,
                   dot_color=dc_only_gene,
                   dot_ax=plt.gca(),
                   cmap='Blues')
plt.gca().set_ylabel('Leiden cluster')

# add nice little dividing lines to guide the eye
a = 0
for x in [2, 4, 6, 8, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47]:
    plt.plot([x + a + 0.5, x + a + 0.5], [0, 15], color='lightgray', lw=1)
    a += 1

# savefig('pbmc5k_citeseq_dotplot_cellbender_0.1.pdf')

plt.show()

# size legend
plt.figure(figsize=(2, 1))
dp._plot_size_legend(plt.gca())
# savefig('pbmc5k_citeseq_dotplot_cellbender_0.1_dot_size_legend.pdf')
plt.show()

# colorbars
plt.figure(figsize=(2, 0.2))
dp.cmap = 'Reds'
dp._plot_colorbar(plt.gca(), normalize_ab)
# savefig('pbmc5k_citeseq_dotplot_cellbender_0.1_primary_colorbar.pdf')
plt.show()

plt.figure(figsize=(2, 0.2))
dp.cmap = 'Blues'
dp._plot_colorbar(plt.gca(), normalize_gene)
# savefig('pbmc5k_citeseq_dotplot_cellbender_0.1_secondary_colorbar.pdf')
plt.show()

sc.set_figure_params(fontsize=12, vector_friendly=True)

### a. CITEseq CellRanger

In [ ]:
# do the dotplot computation using scanpy

sc.set_figure_params(fontsize=18, vector_friendly=True)

dp = sc.pl.dotplot(
    adata_qc,
    var_names=matched_antibody_features_expressed,
    groupby='leiden',
    layer='cellranger',
    log=True,
    vmin=-0.5,
    color_map='Blues',
    colorbar_title='Log expression\nin group',
    var_group_rotation=0,
    dot_min=0,
    dot_max=1,
    show=True,
    return_fig=True,
)

# dataframes of dot size and dot color
dc = dp.dot_color_df.copy()
ds = dp.dot_size_df.copy()

# add in empty columns at the appropriate places to split things up
for i in reversed([2, 4, 6, 8, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47]):
    ds.insert(loc=i, column='', value=np.nan, allow_duplicates=True)
    dc.insert(loc=i, column='', value=np.nan, allow_duplicates=True)

col_ab = [c for c in ds.columns.tolist() if c.endswith('_TotalSeqB')]
col_not_ab = [c for c in ds.columns.tolist() if not c.endswith('_TotalSeqB')]
col_gene = [c for c in ds.columns.tolist() if ((not c.endswith('_TotalSeqB')) and (c != ''))]
col_not_gene = [c for c in ds.columns.tolist() if not ((not c.endswith('_TotalSeqB')) and (c != ''))]

# to compute "normalize"
plt.figure(figsize=(7, 4))
normalize_ab, _, _ = make_scanpy_doplot(dot_size=ds[col_ab],
                                        dot_color=dc[col_ab],
                                        dot_ax=plt.gca(),
                                        vmin=0,
                                        vmax=2.21,  # max val
                                        cmap='Reds')
plt.show()

plt.figure(figsize=(7, 4))
normalize_gene, _, _ = make_scanpy_doplot(dot_size=ds[col_gene],
                                          dot_color=dc[col_gene],
                                          dot_ax=plt.gca(),
                                          vmin=0,
                                          vmax=1.44,  # max val
                                          cmap='Blues')
plt.show()

# the combined plot
plt.figure(figsize=(20, 4))
ds_only_ab = ds.copy()
ds_only_ab[col_not_ab] = 0
dc_only_ab = dc.copy()
dc_only_ab[col_not_ab] = 0
make_scanpy_doplot(dot_size=ds_only_ab,
                   dot_color=dc_only_ab,
                   dot_ax=plt.gca(),
                   cmap='Reds')

ds_only_gene = ds.copy()
ds_only_gene[col_not_gene] = 0
dc_only_gene = dc.copy()
dc_only_gene[col_not_gene] = 0
make_scanpy_doplot(dot_size=ds_only_gene,
                   dot_color=dc_only_gene,
                   dot_ax=plt.gca(),
                   cmap='Blues')
plt.gca().set_ylabel('Leiden cluster')

# add nice little dividing lines to guide the eye
a = 0
for x in [2, 4, 6, 8, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47]:
    plt.plot([x + a + 0.5, x + a + 0.5], [0, 15], color='lightgray', lw=1)
    a += 1

# savefig('pbmc5k_citeseq_dotplot_raw.pdf')

plt.show()

# size legend
plt.figure(figsize=(2, 1))
dp._plot_size_legend(plt.gca())
# savefig('pbmc5k_citeseq_dotplot_raw_dot_size_legend.pdf')
plt.show()

# colorbars
plt.figure(figsize=(2, 0.2))
dp.cmap = 'Reds'
dp.color_legend_title = 'Log antibody counts'
dp._plot_colorbar(plt.gca(), normalize_ab)
# savefig('pbmc5k_citeseq_dotplot_raw_primary_colorbar.pdf')
plt.show()

plt.figure(figsize=(2, 0.2))
dp.cmap = 'Blues'
dp.color_legend_title = 'Log gene counts'
dp._plot_colorbar(plt.gca(), normalize_gene)
# savefig('pbmc5k_citeseq_dotplot_raw_secondary_colorbar.pdf')
plt.show()

sc.set_figure_params(fontsize=12, vector_friendly=True)

### c, d. CITEseq Pearson correlations

In [ ]:
raw_correlation_threshold = 1
raw_gene_expression_threshold = 0.2

In [ ]:
adata_qc

In [ ]:
# gather a dataframe with all the information we need

genes = matched_antibody_features[0::2]
antibodies = matched_antibody_features[1::2]

def get_feature_counts(adata, layer, feature) -> np.ndarray:
    """Obtain feature counts as a vector"""
    return np.array(adata.layers[layer][:, adata.var_names == feature].todense()).squeeze()

df_counts = adata_qc.obs[['leiden']].copy()

for feature in genes + antibodies:
    df_counts[feature + '_raw'] = get_feature_counts(adata_qc, layer='cellranger', feature=feature)
    df_counts[feature] = get_feature_counts(adata_qc, layer='cellbender', feature=feature)

df_log_counts = adata_qc.obs[['leiden']].copy()

for feature in genes + antibodies:
    df_log_counts[feature + '_raw'] = np.log1p(get_feature_counts(adata_qc, layer='cellranger', feature=feature))
    df_log_counts[feature] = np.log1p(get_feature_counts(adata_qc, layer='cellbender', feature=feature))

# feature means across clusters
df_frac_expressing = df_counts.groupby('leiden').agg(lambda x: np.mean(x > 0))

# feature means across clusters
df_means = df_log_counts.groupby('leiden').mean()

# standard error of the mean
df_std_means = (df_log_counts.groupby('leiden').std()
                / df_log_counts.groupby('leiden').count().apply(np.sqrt))
df_means

In [ ]:
class GridPlot:

    def __init__(self,
                 grid_shape,
                 figsize,
                 xlabel,
                 ylabel):
        assert grid_shape is not None, 'Must initialize with a grid_shape'
        self.fig = plt.figure(figsize=figsize)
        print('Grid plot initialized')
        self.grid_shape = grid_shape
        self.xlabel = xlabel
        self.ylabel = ylabel
        self.axes = []
        self._i = 1

    def add_plot(self):
        """Command for creating a grid plot piecemeal"""

        is_left_edge = ((self._i - 1) % self.grid_shape[1] == 0)
        is_bottom_row = (self._i - self.grid_shape[1] > 0) or (self.grid_shape[0] == 1)  # currently

        ax = self.fig.add_subplot(self.grid_shape[0], self.grid_shape[1], self._i)
        self.axes.append({'ax': ax, 'is_left_edge': is_left_edge, 'is_bottom_row': is_bottom_row})

        if is_left_edge:
            ax.set_ylabel(self.ylabel)
        if is_bottom_row:
            ax.set_xlabel(self.xlabel)
            self.axes[self._i - self.grid_shape[1] - 1]['ax'].set_xlabel('')
            self.axes[self._i - self.grid_shape[1] - 1]['is_bottom_row'] = False

        self._i += 1

In [ ]:
def fit_taking_xerr_into_account(x, y, xerr):
    """Fit a line using weighted OLS"""
    fit = np.polyfit(x, y, deg=1)
    m = fit[0]
    b = fit[1]
    induced_yerr = xerr * m
    weights = 1./(induced_yerr + 0.01)
    fit = np.polyfit(x, y, w=weights, deg=1)
    m = fit[0]
    b = fit[1]
    return (m, b)


def scale_data(x, y, xerr, yerr):
    """Apply scaling to collapse to line"""

    # get data on same relative scale
    x_rescaled = x / np.std(x)
    xerr_rescaled = xerr / np.std(x)
    y_rescaled = y / np.std(y)
    yerr_rescaled = yerr / np.std(y)

    # make slope 1
    m, _ = fit_taking_xerr_into_account(x_rescaled, y_rescaled, xerr_rescaled)
    y_rescaled = y_rescaled / m
    yerr_rescaled = yerr_rescaled / m

    return x_rescaled, y_rescaled, xerr_rescaled, yerr_rescaled


def make_gridplot(df_means,
                  df_std_means,
                  skip_abs,
                  use_raw,
                  scaled,
                  hard_ymax=None,
                  hard_xmax=None,
                  grid_shape=(6, 7), figsize=(12, 12), ms=10):

    cmap = cc.glasbey

    grid = GridPlot(
        grid_shape=grid_shape,
        figsize=figsize,
        xlabel=('Scaled ' if scaled else '') + 'log1p RNA',
        ylabel=('Scaled ' if scaled else '') + 'log1p Antibody',
    )  # initialize

    antibodies = [c for c in df_means.columns
                  if (('TotalSeqB' in c) and ('_raw' not in c))]
    lookup_gene_dict = dict(zip(matched_antibody_features[1::2], matched_antibody_features[::2]))
    print(lookup_gene_dict)
    genes = [lookup_gene_dict[a] for a in antibodies]

    abs_with_low_correlation = []
    abs_with_low_gene_expression = []

    for i, (g, p, c) in enumerate(zip(genes, antibodies, cmap)):

        if p in skip_abs:
            continue

        x = df_means[g + ('_raw' if use_raw else '')]
        x_err = df_std_means[g + ('_raw' if use_raw else '')]
        y = df_means[p + ('_raw' if use_raw else '')]
        y_err = df_std_means[p + ('_raw' if use_raw else '')]

        if x.max() < raw_gene_expression_threshold:
            abs_with_low_gene_expression.append(p)

        if scaled:
            x, y, x_err, y_err = scale_data(x, y, x_err, y_err)

        grid.add_plot()
        plt.errorbar(x=x, y=y, xerr=x_err, yerr=y_err,
                     linestyle='', marker='.', ms=ms, color=c)
        m, b = fit_taking_xerr_into_account(x, y, x_err)
        plt.plot([0, x.max()], [b, b + m * x.max()], '-', color=c, alpha=0.75, lw=0.5)
        plt.title(p)
        plt.xlim(left=0)
        xlims = plt.gca().get_xlim()
        plt.xlim(left=-1 * (xlims[1] - xlims[0]) / 20)
        if hard_xmax is not None:
            plt.xlim(right=hard_xmax)
        plt.ylim(bottom=0)
        ylims = plt.gca().get_ylim()
        plt.ylim(bottom=-1 * (ylims[1] - ylims[0]) / 20)
        if hard_ymax is not None:
            plt.ylim(top=hard_ymax)
        plt.grid(False)

#         print(f'{p}: correlation {m}')
        if m <= raw_correlation_threshold:
            abs_with_low_correlation.append(p)

    if scaled:
        # make all axis limits the same
        xmax = 0
        ymax = 0
        for axis in grid.axes:
            ax = axis['ax']
            xmax = ax.get_xlim()[1] if (ax.get_xlim()[1] > xmax) else xmax
            ymax = ax.get_ylim()[1] if (ax.get_ylim()[1] > ymax) else ymax
        if hard_xmax is not None:
            xmax = hard_xmax
        if hard_ymax is not None:
            ymax = hard_ymax
        for axis in grid.axes:
            ax = axis['ax']
            ax.set_xlim([-1 * xmax / 20, xmax])
            ax.set_ylim([-1 * ymax / 20, ymax])
            if not axis['is_left_edge']:
                ticks = ax.get_yticks()
                ax.set_yticklabels(['' for t in ticks])
            if not axis['is_bottom_row']:
                ticks = ax.get_xticks()
                ax.set_xticklabels(['' for t in ticks])
    plt.tight_layout()

    return abs_with_low_correlation, abs_with_low_gene_expression, grid


def make_nongridplot(df_means,
                     df_std_means,
                     skip_abs,
                     use_raw,
                     ms=10):

    cmap = cc.glasbey

    antibodies = [c for c in df_means.columns
                  if (('TotalSeqB' in c) and ('_raw' not in c))]
    lookup_gene_dict = dict(zip(matched_antibody_features[1::2], matched_antibody_features[::2]))
    print(lookup_gene_dict)
    genes = [lookup_gene_dict[a] for a in antibodies]

    all_x = []
    all_y = []

    for i, (g, p, c) in enumerate(zip(genes, antibodies, cmap)):

        if p in skip_abs:
            continue

        x = df_means[g + ('_raw' if use_raw else '')]
        x_err = df_std_means[g + ('_raw' if use_raw else '')]
        y = df_means[p + ('_raw' if use_raw else '')]
        y_err = df_std_means[p + ('_raw' if use_raw else '')]

        x, y, x_err, y_err = scale_data(x, y, x_err, y_err)
        all_x.extend(x)
        all_y.extend(y)

        plt.errorbar(x=x, y=y, xerr=x_err, yerr=y_err,
                     linestyle='', marker='o', ms=ms, color=c,
                     elinewidth=0.5, label=p.replace('_TotalSeqB', ''),
                     markeredgecolor='none', alpha=0.9)
#         plt.plot(x, y, '.', ms=ms, color=c)
#         m, b = fit_taking_xerr_into_account(x, y, x_err)
#         plt.plot([0, 4.5], [b, m * 4.5 + b], '-', color=c)

    plt.draw()

    fit = np.polyfit(all_x, all_y, deg=1)
    m = fit[0]
    b = fit[1]
    plt.plot([0, 4.5], [b, m * 4.5 + b], 'k-', lw=3)

    plt.xlabel('Scaled log1p RNA count per cluster')
    plt.ylabel('Scaled log1p Antibody count per cluster')
    plt.title('Raw' if use_raw else 'CellBender')

    plt.grid(False)
    plt.tight_layout()

In [ ]:
abs_low_corr, abs_low_mrna, grid = make_gridplot(df_means, df_std_means, skip_abs=[],
                                                 grid_shape=(4, 8), figsize=(15, 8),
                                                 use_raw=True, scaled=False)
for axis_dict in grid.axes:
    axis_dict['ax'].set_title(axis_dict['ax'].get_title().replace('_TotalSeqB', ''))

# savefig('pbmc5k_citeseq_modalities_scatterplot_per_antibody_raw.pdf')

plt.show()

In [ ]:
abs_low_corr

In [ ]:
abs_low_mrna

In [ ]:
_, _, grid = make_gridplot(
    df_means, df_std_means,
    skip_abs=[], grid_shape=(4, 8), figsize=(15, 8),
    use_raw=False, scaled=False,
)

for axis_dict in grid.axes:
    axis_dict['ax'].set_title(axis_dict['ax'].get_title().replace('_TotalSeqB', ''))

# savefig('pbmc5k_citeseq_modalities_scatterplot_per_antibody.pdf')

plt.show()

In [ ]:
sc.set_figure_params(fontsize=14, vector_friendly=True)

In [ ]:
plt.figure(figsize=(4, 8))

plt.subplot(2, 1, 1)

make_nongridplot(
    df_means,
    df_std_means,
    skip_abs=abs_low_corr + abs_low_mrna,
    use_raw=True,
    ms=5,
)
plt.ylim(top=6.5)
plt.xticks([0, 1, 2, 3, 4], labels=['', '', '', '', ''])
plt.xlabel('')

plt.subplot(2, 1, 2)

make_nongridplot(
    df_means,
    df_std_means,
    skip_abs=abs_low_corr + abs_low_mrna,
    use_raw=False,
    ms=5,
)
plt.ylim(top=6.5)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

# savefig('pbmc5k_citeseq_modalities_scatterplot.pdf')

plt.show()

In [ ]:
def get_pearson_correlations(df, antibodies, matched_antibody_features):

    corr = []
    raw_corr = []
    labels = []

    for p in antibodies:
        y = df[p].values
        gene = matched_antibody_features[np.where(np.array(matched_antibody_features) == p)[0].item() - 1]
        x = df[gene].values
        print(f'{gene}: {p}')
        corr.append(corr_pearson(x, y)[0])

        y_raw = df[p + '_raw'].values
        x_raw = df[gene + '_raw'].values
        raw_corr.append(corr_pearson(x_raw, y_raw)[0])

        labels.append(p.replace('_TotalSeqB', ''))

    return labels, raw_corr, corr

In [ ]:
xticklabels, raw_corr, cb_corr = get_pearson_correlations(
    df=df_frac_expressing,
    antibodies=antibodies,
    matched_antibody_features=matched_antibody_features,
)

In [ ]:
plt.figure(figsize=(3, 8))

width = 0.2
xticks = np.arange(len(raw_corr))

plt.barh(y=xticks + width / 1.5, width=np.maximum(0, raw_corr)[::-1],
         height=width, color='black', label='Raw')
plt.barh(y=xticks - width / 1.5, width=np.maximum(0, cb_corr)[::-1],
         height=width, color='tab:cyan', label='CellBender')

plt.plot([0, 0], [-1, len(xticks)], color='lightgray', lw=0.75)
plt.grid(False)
plt.yticks(ticks=xticks, labels=xticklabels[::-1])
plt.ylim([-1, len(xticks)])
plt.xlim([-0.1, 1.1])
plt.xlabel('Pearson correlation between\nfraction of cells per cluster\nexpressing antibody and RNA')
plt.legend(fontsize=10, loc='center left', bbox_to_anchor=(1, 0.5))
plt.ylabel('Antibody')

# savefig('pbmc5k_citeseq_modalities_pearson_frac.pdf')

plt.show()